# Preprocessing: Merge variables
The mock network is intended for testing algorithms without the need of setting up an 
entire vantage6 network.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'

## MockClient
We mock to have two organizations with three databases each (<code>rps_cohort</code>, <code>pelvis_cohort</code> and <code>rps_pelvis_cohort</code>). <br><br>
The first organization has:
* <code>rps_cohort[:10]</code> (10 pts)
* <code>pelvis_cohort[:10]</code> (10 pts)
* <code>rps_pelvis_cohort[:10]</code> (10 pts)

The second organization has:
* <code>rps_cohort[10:]</code> (304 pts)
* <code>pelvis_cohort[10:]</code> (276 pts)
* <code>non_liposarcoma_cohort[10:]</code> (590 pts)

In [ ]:
# Load the dataframes from parquet files
rps_cohort = pd.read_parquet("data/rps_cohort.parquet")
pelvis_cohort = pd.read_parquet("data/pelvis_cohort.parquet")
rps_pelvis_cohort = pd.read_parquet("data/rps_pelvis_cohort.parquet")


In [ ]:
from vantage6.algorithm.mock.network import MockNetwork

network = MockNetwork(
    "v6_preprocessing",
    datasets=[
        {
            "rps": {"database": rps_cohort[:10], "db_type": "omop"},
            "pelvis": {"database": pelvis_cohort[:10], "db_type": "omop"},
            "rps_pelvis": {"database": rps_pelvis_cohort[:10], "db_type": "omop"}
        },
        {
            "rps": {"database": rps_cohort[10:], "db_type": "omop"},
            "pelvis": {"database": pelvis_cohort[10:], "db_type": "omop"},
            "rps_pelvis": {"database": rps_pelvis_cohort[10:], "db_type": "omop"}
        }
    ],
    collaboration_id=1,
)

client = network.user_client

In [ ]:
client.dataframe.preprocess(
    id_=1,
    method="merge_variables",
    image="v6-preprocessing",
    arguments={
        "column1": "histology",
        "column2": "sex",
        "output_column": "histology_sex_merged"
    }
)

In [ ]:
client.network.get_node(1).dataframes["rps"]